In [64]:
import cv2
import json
import numpy as np
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm

In [65]:
json_path = '../dataset/WLASL_v0.3.json'
try:
    with open(json_path, 'r') as file:
        wlasl_data = json.load(file)
    print(f'Total words in dataset: {len(wlasl_data)}')
    print('First 10 words found:')
    for i in range(10):
        print(f"- {wlasl_data[i]['gloss']}")
except FileNotFoundError:
    print('Could not find the JSON file. Check the folder path and name.')

Total words in dataset: 2000
First 10 words found:
- book
- drink
- computer
- before
- chair
- go
- clothes
- who
- candy
- cousin


### MediaPipe Holistic Landmarks

| Part | Landmarks | Values | Used? |
|---|---|---|---|
| Face | 468 × (x,y,z) | 1404 | ❌ Excluded — adds noise, not needed for ASL |
| Pose | 33 × (x,y,z,vis) | 132 | ✅ Upper body context |
| Left hand | 21 × (x,y,z) | 63 | ✅ Primary signal |
| Right hand | 21 × (x,y,z) | 63 | ✅ Primary signal |

**Total per frame: `pose(132) + left_hand(63) + right_hand(63) = 258`**

Face is intentionally excluded — it adds 1404 noisy features with minimal ASL discriminative value and significantly inflates model size.

In [66]:
with open('../dataset/nslt_100.json') as f:
    NSLT = json.load(f)

with open('../dataset/WLASL_v0.3.json') as f:
    WLASL = json.load(f)

In [67]:
# video_id → gloss metadata (fps, trim points, bbox)
WLASL_META = {}
for entry in WLASL:
    for inst in entry['instances']:
        WLASL_META[inst['video_id']] = {
            'gloss'      : entry['gloss'],
            'fps'        : inst['fps'],
            'frame_start': inst['frame_start'],
            'frame_end'  : inst['frame_end'],
            'bbox'       : inst['bbox'],
        }

# class_idx → gloss name
CLASS_TO_GLOSS = {}
for vid_id, val in NSLT.items():
    cls = val['action'][0]
    if vid_id in WLASL_META and cls not in CLASS_TO_GLOSS:
        CLASS_TO_GLOSS[cls] = WLASL_META[vid_id]['gloss']

GLOSS_TO_CLASS = {v: k for k, v in CLASS_TO_GLOSS.items()}

In [68]:
VIDEO_DIR    = '../dataset/videos/'
OUTPUT_DIR   = '../dataset/processed/'
NUM_FRAMES   = 30
NUM_CLASSES  = 100
NUM_FEATURES = 258
FPS          = 25
MIN_FRAMES   = 10
USE_BBOX     = True
MEDIAPIPE_INPUT_SIZE = (256, 256)

In [69]:
TRAIN_IDS = [vid for vid, val in NSLT.items() if val['subset'] == 'train']
VAL_IDS   = [vid for vid, val in NSLT.items() if val['subset'] == 'val']
TEST_IDS  = [vid for vid, val in NSLT.items() if val['subset'] == 'test']

print(f'Classes : {NUM_CLASSES}')
print(f'Train   : {len(TRAIN_IDS)} | Val: {len(VAL_IDS)} | Test: {len(TEST_IDS)}')
print(f'Total   : {len(NSLT)} videos')
print(f'Sample class map: { {k: CLASS_TO_GLOSS[k] for k in range(10)} }')

Classes : 100
Train   : 1442 | Val: 338 | Test: 258
Total   : 2038 videos
Sample class map: {0: 'book', 1: 'drink', 2: 'computer', 3: 'before', 4: 'chair', 5: 'go', 6: 'clothes', 7: 'who', 8: 'candy', 9: 'cousin'}


In [70]:
def extract_keypoints(result) -> np.ndarray:
    pose = np.array(
        [[lm.x, lm.y, lm.z, lm.visibility] for lm in result.pose_landmarks],
        dtype=np.float32
    ).flatten() if result.pose_landmarks else np.zeros(132, dtype=np.float32)

    lh = np.array(
        [[lm.x, lm.y, lm.z] for lm in result.left_hand_landmarks],
        dtype=np.float32
    ).flatten() if result.left_hand_landmarks else np.zeros(63, dtype=np.float32)

    rh = np.array(
        [[lm.x, lm.y, lm.z] for lm in result.right_hand_landmarks],
        dtype=np.float32
    ).flatten() if result.right_hand_landmarks else np.zeros(63, dtype=np.float32)

    return np.concatenate([pose, lh, rh])  # (258,)

In [71]:
def normalize_keypoints(keypoints: np.ndarray) -> np.ndarray:
    kp = keypoints.copy()

    # ── Pose: normalize relative to mid-shoulder ──────────────────────────────
    # MediaPipe pose indices: 11=left shoulder, 12=right shoulder
    if not np.allclose(kp[0:132], 0):
        pose = kp[0:132].reshape(33, 4)           # (33, x/y/z/vis)
        mid_shoulder = (pose[11, :3] + pose[12, :3]) / 2.0
        pose[:, :3] -= mid_shoulder               # translate
        span = np.linalg.norm(pose[:, :3].max(axis=0) - pose[:, :3].min(axis=0)) + 1e-6
        pose[:, :3] /= span                       # unit-scale (visibility left as-is)
        kp[0:132] = pose.flatten()

    # ── Left hand: wrist-relative + unit-scale ────────────────────────────────
    if not np.allclose(kp[132:195], 0):
        lh   = kp[132:195].reshape(21, 3)
        lh  -= lh[0]                              # wrist is landmark 0
        span = np.linalg.norm(lh.max(axis=0) - lh.min(axis=0)) + 1e-6
        kp[132:195] = (lh / span).flatten()

    # ── Right hand: wrist-relative + unit-scale ───────────────────────────────
    if not np.allclose(kp[195:258], 0):
        rh   = kp[195:258].reshape(21, 3)
        rh  -= rh[0]
        span = np.linalg.norm(rh.max(axis=0) - rh.min(axis=0)) + 1e-6
        kp[195:258] = (rh / span).flatten()

    return kp

In [72]:
def process_video(video_id: str, landmarker) -> np.ndarray | None:
    meta       = WLASL_META.get(video_id)
    video_path = os.path.join(VIDEO_DIR, f'{video_id}.mp4')

    if meta is None or not os.path.exists(video_path):
        return None

    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total == 0:
        cap.release()
        return None

    f_start  = meta['frame_start'] - 1
    f_end    = total if meta['frame_end'] == -1 else min(meta['frame_end'], total)
    n_usable = f_end - f_start

    if n_usable < MIN_FRAMES:
        cap.release()
        return None

    indices         = np.linspace(f_start, f_end - 1, NUM_FRAMES, dtype=int)
    x1, y1, x2, y2 = meta['bbox']

    sequence = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()

        if not ret or frame is None:
            sequence.append(sequence[-1] if sequence else np.zeros(NUM_FEATURES, dtype=np.float32))
            continue

        if USE_BBOX:
            h, w    = frame.shape[:2]
            cx1     = max(0, min(x1, w - 1))
            cy1     = max(0, min(y1, h - 1))
            cx2     = max(cx1 + 1, min(x2, w))
            cy2     = max(cy1 + 1, min(y2, h))
            cropped = frame[cy1:cy2, cx1:cx2]
            frame   = cropped if cropped.size > 0 else frame

        frame = cv2.resize(frame, MEDIAPIPE_INPUT_SIZE)

        rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result   = landmarker.detect(mp_image)

        kp = extract_keypoints(result)
        kp = normalize_keypoints(kp)
        sequence.append(kp)

    cap.release()
    return np.array(sequence, dtype=np.float32)  # (30, 258)

In [73]:
def build_dataset():
    options = vision.HolisticLandmarkerOptions(
        base_options=python.BaseOptions(
            model_asset_path='../models/holistic_landmarker.task'
        ),
        running_mode=vision.RunningMode.IMAGE,
        min_face_detection_confidence=0.5,
        min_pose_detection_confidence=0.5,
        min_hand_landmarks_confidence=0.5,
        output_segmentation_mask=False,
    )

    skipped   = 0
    processed = 0

    with vision.HolisticLandmarker.create_from_options(options) as landmarker:
        for video_id, val in tqdm(NSLT.items(), desc='Extracting keypoints'):
            class_idx = val['action'][0]
            gloss     = CLASS_TO_GLOSS[class_idx]
            subset    = val['subset']

            out_dir  = os.path.join(OUTPUT_DIR, subset, gloss)
            out_path = os.path.join(out_dir, f'{video_id}.npy')
            os.makedirs(out_dir, exist_ok=True)

            if os.path.exists(out_path):
                processed += 1
                continue

            seq = process_video(video_id, landmarker)
            if seq is not None:
                np.save(out_path, seq)
                processed += 1
            else:
                skipped += 1

    print(f'Done. Processed: {processed} | Skipped: {skipped}')


build_dataset()

Extracting keypoints: 100%|██████████| 2038/2038 [00:00<00:00, 2807.05it/s]

Done. Processed: 1906 | Skipped: 132


## Consolidate .npy files → train / val / test .npz arrays

In [74]:
def consolidate_splits(output_dir: str, class_to_gloss: dict) -> dict:
    gloss_to_class = {v: k for k, v in class_to_gloss.items()}
    data = {'train': {'X': [], 'y': []},
            'val':   {'X': [], 'y': []},
            'test':  {'X': [], 'y': []}}

    for subset in ['train', 'val', 'test']:
        subset_dir = os.path.join(output_dir, subset)
        if not os.path.isdir(subset_dir):
            print(f'Warning: {subset_dir} not found — skipping.')
            continue

        for gloss in os.listdir(subset_dir):
            gloss_dir = os.path.join(subset_dir, gloss)
            if not os.path.isdir(gloss_dir):
                continue
            label = gloss_to_class.get(gloss)
            if label is None:
                print(f'Warning: gloss "{gloss}" not in CLASS_TO_GLOSS — skipping.')
                continue

            for npy_file in os.listdir(gloss_dir):
                if not npy_file.endswith('.npy'):
                    continue
                seq = np.load(os.path.join(gloss_dir, npy_file))
                if seq.shape != (NUM_FRAMES, NUM_FEATURES):
                    print(f'Shape mismatch {npy_file}: {seq.shape} — skipping.')
                    continue
                data[subset]['X'].append(seq)
                data[subset]['y'].append(label)

    # Stack and save
    for subset, d in data.items():
        if not d['X']:
            print(f'No data for split "{subset}" — skipping.')
            continue
        X = np.array(d['X'], dtype=np.float32)   # (N, 30, 258)
        y = np.array(d['y'], dtype=np.int32)      # (N,)
        out = os.path.join(output_dir, f'{subset}.npz')
        np.savez_compressed(out, X=X, y=y)
        size_mb = os.path.getsize(out) / 1e6
        print(f'Saved {subset}.npz  X={X.shape}  y={y.shape}  ({size_mb:.1f} MB)')
        d['X'], d['y'] = X, y

    return data


splits = consolidate_splits(OUTPUT_DIR, CLASS_TO_GLOSS)

# Save label map for inference
with open(os.path.join(OUTPUT_DIR, 'label_to_gloss.json'), 'w') as f:
    json.dump({str(k): v for k, v in CLASS_TO_GLOSS.items()}, f, indent=2)
print(f'Saved label_to_gloss.json ({len(CLASS_TO_GLOSS)} classes)')

Saved train.npz  X=(1396, 30, 258)  y=(1396,)  (15.4 MB)
Saved val.npz  X=(320, 30, 258)  y=(320,)  (3.4 MB)
Saved test.npz  X=(190, 30, 258)  y=(190,)  (2.3 MB)
Saved label_to_gloss.json (100 classes)


## Verification — sanity-check the saved arrays

In [ ]:
print('=== Dataset Verification ===')
for subset in ['train', 'val', 'test']:
    d = np.load(os.path.join(OUTPUT_DIR, f'{subset}.npz'))
    X, y = d['X'], d['y']
    print(f'{subset:5s}  X={X.shape}  y={y.shape}  '
          f'range=[{X.min():.3f}, {X.max():.3f}]  '
          f'NaN={np.isnan(X).sum()}  '
          f'classes={len(np.unique(y))}')

# Check a single sample
train_d = np.load(os.path.join(OUTPUT_DIR, 'train.npz'))
with open(os.path.join(OUTPUT_DIR, 'label_to_gloss.json')) as f:
    lbl_map = json.load(f)

sample_x = train_d['X'][0]
sample_y = int(train_d['y'][0])
print(f'\nSample[0]: label={sample_y}  gloss="{lbl_map[str(sample_y)]}"  shape={sample_x.shape}')
print(f'  Pose  [{0}:{132}]      mean={sample_x[:, 0:132].mean():.4f}')
print(f'  LHand [{132}:{195}]  mean={sample_x[:, 132:195].mean():.4f}')
print(f'  RHand [{195}:{258}]  mean={sample_x[:, 195:258].mean():.4f}')
print('\n✓ Preprocessing complete — ready for model training!')

=== Dataset Verification ===
train  X=(1396, 30, 258)  y=(1396,)  range=[-0.987, 1.000]  NaN=0  classes=100
val    X=(320, 30, 258)  y=(320,)  range=[-0.982, 1.000]  NaN=0  classes=100
test   X=(190, 30, 258)  y=(190,)  range=[-0.980, 1.000]  NaN=0  classes=96

Sample[0]: label=51  gloss="accident"  shape=(30, 258)
  Pose  [0:132]      mean=0.1577
  LHand [132:195]  mean=-0.0650
  RHand [195:258]  mean=-0.0292

✓ Preprocessing complete — ready for model training!


## Quick loader — copy this into your training notebook

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  COPY THIS BLOCK INTO YOUR TRAINING NOTEBOOK
# ════════════════════════════════════════════════════════════════════
import json, numpy as np

OUTPUT_DIR = '../dataset/processed/'

train = np.load(OUTPUT_DIR + 'train.npz')
val   = np.load(OUTPUT_DIR + 'val.npz')
test  = np.load(OUTPUT_DIR + 'test.npz')

X_train, y_train = train['X'], train['y']   # (1442, 30, 258)
X_val,   y_val   = val['X'],   val['y']     # (338,  30, 258)
X_test,  y_test  = test['X'],  test['y']    # (258,  30, 258)

with open(OUTPUT_DIR + 'label_to_gloss.json') as f:
    label_to_gloss = {int(k): v for k, v in json.load(f).items()}

NUM_CLASSES  = len(label_to_gloss)          # 100
SEQ_LEN      = X_train.shape[1]             # 30
NUM_FEATURES = X_train.shape[2]             # 258

print(f'X_train {X_train.shape}  y_train {y_train.shape}')
print(f'X_val   {X_val.shape}    y_val   {y_val.shape}')
print(f'X_test  {X_test.shape}   y_test  {y_test.shape}')
print(f'Classes={NUM_CLASSES}  seq_len={SEQ_LEN}  features={NUM_FEATURES}')

X_train (1396, 30, 258)  y_train (1396,)
X_val   (320, 30, 258)    y_val   (320,)
X_test  (190, 30, 258)   y_test  (190,)
Classes=100  seq_len=30  features=258
